# BioAuth Lab: Behavioural Biometrics Data Science & ML Analysis
### End-to-End Evaluation of Keystroke, Mouse, and Touch Dynamics with Multimodal Fusion

Behavioural biometrics authenticates individuals based on **how they interact** with input peripherals rather than static credentials (passwords, tokens). This notebook explores:
1. **Feature Engineering** from raw keystroke timestamps, mouse trajectories, and touch swipes.
2. **Classifier Benchmarking**: Comparing One-Class SVM, Isolation Forest, Random Forest, KNN, and Gradient Boosting.
3. **Improvement Pipeline**: Evaluating the impact of outlier removal, feature scaling, mutual information selection, and GridSearchCV tuning.
4. **Score-Level Multimodal Fusion**: Finding optimal convex weights to minimize Equal Error Rate (EER).

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set project root for importing bioauth core modules
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.db import get_session, User, Sample
from core.models import load_dataset, compare_models, BiometricPipeline, MultimodalFusionSystem
from core.metrics import evaluate_predictions, compute_eer

print("BioAuth Lab analysis environment loaded successfully.")

## 1. Load Biometric Dataset from SQLite
We query the seeded database (`data/bioauth.db`) containing raw and extracted features for 20 users across Keystroke, Mouse, and Touch modalities.

In [ ]:
session = get_session()
user = session.query(User).filter_by(username="user_01").first()
user_id = user.id

df_k, y_k, feats_k = load_dataset(user_id=user_id, modality="keystroke")
df_m, y_m, feats_m = load_dataset(user_id=user_id, modality="mouse")
df_t, y_t, feats_t = load_dataset(user_id=user_id, modality="touch")
session.close()

print(f"User {user.username} (ID: {user_id})")
print(f"- Keystroke dataset: {df_k.shape[0]} samples, {df_k.shape[1]} features (Genuine: {sum(y_k==1)}, Impostor: {sum(y_k==0)})")
print(f"- Mouse dataset:     {df_m.shape[0]} samples, {df_m.shape[1]} features (Genuine: {sum(y_m==1)}, Impostor: {sum(y_m==0)})")
print(f"- Touch dataset:     {df_t.shape[0]} samples, {df_t.shape[1]} features (Genuine: {sum(y_t==1)}, Impostor: {sum(y_t==0)})")

## 2. Exploratory Data Analysis (EDA)
Let us inspect the distributions of genuine vs. impostor behavioral features.

In [ ]:
# Inspect Keystroke Dwell and Flight times
eda_k = df_k.copy()
eda_k["label"] = np.where(y_k == 1, "Genuine", "Impostor")

display_cols = ["dwell_mean", "dwell_std", "flight_ud_mean", "flight_dd_mean", "typing_speed_wpm"]
print("Keystroke Features Summary by Class:")
eda_k.groupby("label")[display_cols].agg(["mean", "std"])

In [ ]:
# Inspect Mouse & Touch Kinematic Features
eda_m = df_m.copy()
eda_m["label"] = np.where(y_m == 1, "Genuine", "Impostor")

print("Mouse Dynamics Kinematic Summary by Class:")
eda_m.groupby("label")[["speed_mean", "accel_mean", "path_efficiency", "click_duration_mean"]].agg(["mean", "std"])

## 3. Candidate Model Benchmarking across Modalities
We evaluate 5 architectures using Stratified 5-Fold Cross Validation:
- **One-Class SVM** (Semi-supervised novelty detection)
- **Isolation Forest** (Ensemble tree isolation anomaly detection)
- **Random Forest** (Supervised ensemble with balanced class weighting)
- **K-Nearest Neighbors** (Distance-weighted local density)
- **Gradient Boosting** (Sequential tree boosting)

In [ ]:
modalities = {"Keystroke": (df_k, y_k), "Mouse": (df_m, y_m), "Touch": (df_t, y_t)}
comp_summaries = {}

for name, (X_df, y) in modalities.items():
    res = compare_models(X_df.values, y, n_splits=5)
    comp_summaries[name] = res
    print(f"\n=== {name.upper()} TOP MODEL: {res['best_model']} (EER = {res['best_eer']:.4f}) ===")
    rows = []
    for m_name, m_res in res["results"].items():
        met = m_res["summary_metrics"]
        rows.append({
            "Model": m_name,
            "Mean EER": m_res["mean_eer"],
            "AUC": met["auc"],
            "Accuracy": met["accuracy"],
            "F1": met["f1"]
        })
    print(pd.DataFrame(rows).to_string(index=False))

## 4. Improvement Pipeline: Outlier Rejection, Scaling & GridSearchCV
We compare raw baseline models against the full preprocessing & tuning pipeline:
$$\text{Raw Training Data} \rightarrow \text{Isolation Forest Outlier Filtering} \rightarrow \text{StandardScaler} \rightarrow \text{SelectKBest (MI)} \rightarrow \text{GridSearchCV (Balanced)}$$

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Benchmark on Mouse Modality
X = df_m.values
y = y_m
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Baseline evaluation
y_base_true, y_base_scores = [], []
for tr, val in skf.split(X, y):
    base = RandomForestClassifier(n_estimators=10, random_state=42).fit(X[tr], y[tr])
    y_base_true.extend(y[val])
    y_base_scores.extend(base.predict_proba(X[val])[:, 1])

m_before = evaluate_predictions(np.array(y_base_true), np.array(y_base_scores))

# Tuned pipeline evaluation
y_pipe_true, y_pipe_scores = [], []
for tr, val in skf.split(X, y):
    pipe = BiometricPipeline(base_model_name="Random Forest").fit(X[tr], y[tr], tune_grid=True)
    y_pipe_true.extend(y[val])
    y_pipe_scores.extend(pipe.predict_proba(X[val])[:, 1])

m_after = evaluate_predictions(np.array(y_pipe_true), np.array(y_pipe_scores))

print("Mouse Improvement Benchmark:")
print(f"- BEFORE Pipeline -> EER: {m_before['eer']:.4f} | AUC: {m_before['auc']:.4f} | Acc: {m_before['accuracy']:.4f}")
print(f"- AFTER Pipeline  -> EER: {m_after['eer']:.4f} | AUC: {m_after['auc']:.4f} | Acc: {m_after['accuracy']:.4f}")
print(f"-> Delta EER: {m_before['eer'] - m_after['eer']:+.4f} (Lower is better)")

## 5. Multimodal Score-Level Fusion
We fuse the probabilistic predictions from all three modalities using convex weighting:
$$S_{\text{fusion}} = w_k \cdot S_k + w_m \cdot S_m + w_t \cdot S_t, \quad \sum w_i = 1$$

In [ ]:
# Train pipelines for each modality
pk = BiometricPipeline(base_model_name="Random Forest").fit(df_k.values, y_k)
pm = BiometricPipeline(base_model_name="Random Forest").fit(df_m.values, y_m)
pt = BiometricPipeline(base_model_name="Random Forest").fit(df_t.values, y_t)

s_k = pk.predict_proba(df_k.values)[:, 1]
s_m = pm.predict_proba(df_m.values)[:, 1]
s_t = pt.predict_proba(df_t.values)[:, 1]

fus_sys = MultimodalFusionSystem()
best_weights, min_eer = fus_sys.fit_weights({"keystroke": s_k, "mouse": s_m, "touch": s_t}, y_k)

fused_scores = best_weights[0] * s_k + best_weights[1] * s_m + best_weights[2] * s_t
fus_eval = evaluate_predictions(y_k, fused_scores)

print(f"Optimal Multimodal Fusion Weights: {best_weights} (Keystroke, Mouse, Touch)")
print(f"Fused Verification Performance:")
print(f"- Equal Error Rate (EER): {fus_eval['eer']:.4f}")
print(f"- Area Under Curve (AUC): {fus_eval['auc']:.4f}")
print(f"- Accuracy:              {fus_eval['accuracy']*100:.1f}%")
print(f"- Precision / Recall:    {fus_eval['precision']:.3f} / {fus_eval['recall']:.3f}")
print(f"- Confusion Matrix:      {fus_eval['confusion_matrix']}")

## 6. Conclusions & Key Findings
1. **Complementary Modalities**: While individual modalities achieve competitive EER, fusing temporal rhythm (Keystroke) with spatial kinematics (Mouse & Touch) yields superior robustness against impersonation.
2. **Outlier Filtering**: Genuine users naturally display minor fatigue or distraction; isolating extreme training outliers significantly tightens acceptance boundaries.
3. **Zero-Trust Continuous Verification**: Behavioral biometrics eliminates single-point authentication failure without imposing repetitive password entry on the end user.